In [1]:
import pandas as pd
import numpy as np
import h3
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

In [2]:
RANDOM_SEED = 42

In [3]:
def metricas(y, pred):
    return {"R2": r2_score(y, pred),
            "MAE": mean_absolute_error(y, pred),
            "RMSE": float(np.sqrt(mean_squared_error(y, pred)))}

In [4]:
def buscar(train_val, target, features, param_dist, cv, n_iter=30):
    rf = RandomForestRegressor(bootstrap=True, n_jobs=-1, random_state=RANDOM_SEED)
    search = RandomizedSearchCV(
        rf, param_dist, n_iter=n_iter, cv=cv,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED, n_jobs=1, verbose=0,
    )
    search.fit(train_val[features], train_val[target])
    return search

In [5]:
def get_folds(df, temps, col_temp, n_folds=3, tam_val=12, gap=0):
    folds = []
    for k in range(n_folds, 0, -1):
        fim_val = len(temps) - tam_val * (k - 1)
        ini_val = fim_val - tam_val
        idx_tr  = np.where(df[col_temp].isin(temps[: ini_val - gap]))[0]
        idx_val = np.where(df[col_temp].isin(temps[ini_val:fim_val]))[0]
        folds.append((idx_tr, idx_val))
    return folds

In [6]:
def metricas_cv(search, train_val, target, features, cv):
    res = cross_validate(
        clone(search.best_estimator_), train_val[features], train_val[target], cv=cv,
        scoring={"R2": "r2", "MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error"},
    )
    return pd.DataFrame({"R2": res["test_R2"], "MAE": -res["test_MAE"], "RMSE": -res["test_RMSE"]},
                        index=[f"fold {i}" for i in range(1, len(cv) + 1)])

In [7]:
def baseline_cv(train_val, target, cv):
    linhas = []
    for tr, va in cv:
        media_cel = train_val.iloc[tr].groupby("h3")[target].mean()
        dv = train_val.iloc[va]
        linhas.append(metricas(dv[target], dv["h3"].map(media_cel).fillna(0)))
    return pd.DataFrame(linhas, index=[f"fold {i}" for i in range(1, len(cv) + 1)])

In [8]:
df_agg = pd.read_csv('datasets/database_processed_h3.csv')

In [9]:
len(df_agg)

136944

In [10]:
def risco_log(d, col, q=0.99):
    lg = np.log1p(d[col])                                           
    teto = lg.groupby(d["periodo"]).transform(lambda s: s.quantile(q))
    return (lg / teto.replace(0, np.nan)).clip(0, 1).fillna(0) 

In [11]:
df_agg['periodo'] = pd.to_datetime(dict(year=df_agg["ANO_BO"], month=df_agg["MES_BO"], day=1))

In [12]:
df_agg['lat'] = df_agg['h3'].map(lambda x: h3.cell_to_latlng(x)[0])

In [13]:
df_agg['long'] = df_agg['h3'].map(lambda x: h3.cell_to_latlng(x)[1])

In [15]:
df_agg["DATA_AUX"] = (pd.to_datetime(dict(year=df_agg['ANO_BO'], month=df_agg['MES_BO'], day=1))
                 + pd.to_timedelta((df_agg['QUINZENA_BO'] == 2).astype(int) * 15, unit="D"))

In [16]:
train_val = df_agg[(df_agg["periodo"] >= "2022-01-01") & (df_agg["periodo"] <= "2025-06-01")].copy()
test = df_agg[(df_agg["periodo"] >= "2025-07-01") & (df_agg["periodo"] <= "2026-06-01")].copy()

In [17]:
for d in (train_val, test):
    d["risco_furto"] = risco_log(d, "n_furtos")
    d["risco_roubo"] = risco_log(d, "n_roubos")
    d['risco'] = risco_log(d, 'n_total')

In [18]:
print("treino+val:", train_val["periodo"].min().date(), "->", train_val["periodo"].max().date(), f"({len(train_val):,} linhas)", f"({round(len(train_val)/len(df_agg)*100,2)}%)")
print("teste     :", test["periodo"].min().date(), "->", test["periodo"].max().date(), f"({len(test):,} linhas)", f"({round(len(test)/len(df_agg)*100,2)}%)")

treino+val: 2022-01-01 -> 2025-06-01 (106,512 linhas) (77.78%)
teste     : 2025-07-01 -> 2026-06-01 (30,432 linhas) (22.22%)


In [19]:
train_val.head()

,h3,n_furtos,n_roubos,n_total,ANO_BO,MES_BO,QUINZENA_BO,periodo,lat,long,DATA_AUX,risco_furto,risco_roubo,risco
0,8781219aaffffff,0,0,0,2022,1,1,2022-01-01,-15.481336,-44.366782,2022-01-01,0.0,0.0,0.0
1,8781219aaffffff,0,0,0,2022,1,2,2022-01-01,-15.481336,-44.366782,2022-01-16,0.0,0.0,0.0
2,8781219aaffffff,0,0,0,2022,2,1,2022-02-01,-15.481336,-44.366782,2022-02-01,0.0,0.0,0.0
3,8781219aaffffff,0,0,0,2022,2,2,2022-02-01,-15.481336,-44.366782,2022-02-16,0.0,0.0,0.0
4,8781219aaffffff,0,0,0,2022,3,1,2022-03-01,-15.481336,-44.366782,2022-03-01,0.0,0.0,0.0


validação e busca por hiperparametros do modelo Baseline

In [20]:
temps = np.sort(train_val['DATA_AUX'].unique())

In [21]:
cv = get_folds(train_val, temps, "DATA_AUX", n_folds=3, tam_val=12, gap=0)

In [22]:
param_dist = {
    "n_estimators":     [200, 300, 500],        # quantas árvores
    "max_depth":        [8, 12, 16, 24, None],  # profundidade máxima de cada árvore
    "min_samples_leaf": [1, 5, 10, 20, 50],     # mínimo de linhas em cada folha
    "max_features":     [0.3, 0.5, 0.7, 1.0],   # fração das entradas que cada divisão considera
    "max_samples":      [0.3, 0.5, 0.8, None],  # fração das linhas usada em cada árvore
}

In [26]:
FEATURES = {
    "com_ano": ["lat", "long", "MES_BO", "QUINZENA_BO", "ANO_BO"],
    "sem_ano": ["lat", "long", "MES_BO", "QUINZENA_BO"],
}

In [27]:
buscas_furto, por_fold_furto, linhas_furto = {}, {}, []

In [28]:
for name, feats in FEATURES.items():
        print(f"buscando: {name}")
        b = buscar(train_val, "risco_furto", feats, param_dist, cv)
        t = metricas_cv(b, train_val, "risco_furto", feats, cv)
        buscas_furto[name], por_fold_furto[name] = b, t
        linhas_furto.append({"modelo": name,
                       "R2": t["R2"].mean(), "MAE": t["MAE"].mean(),
                       "RMSE": t["RMSE"].mean(), "RMSE_desvio_folds": t["RMSE"].std()})

buscando: com_ano
buscando: sem_ano


In [29]:
base_furto = baseline_cv(train_val, "risco_furto", cv)
por_fold_furto["media_celula"] = base_furto
linhas_furto.append({"modelo": "media_celula",
                   "R2": base_furto["R2"].mean(), "MAE": base_furto["MAE"].mean(),
                   "RMSE": base_furto["RMSE"].mean(), "RMSE_desvio_folds": base_furto["RMSE"].std()})

In [30]:
pd.DataFrame(linhas_furto).round(4)

,modelo,R2,MAE,RMSE,RMSE_desvio_folds
0,com_ano,0.9527,0.0412,0.0646,0.0007
1,sem_ano,0.9514,0.0414,0.0654,0.0004
2,media_celula,0.9532,0.0406,0.0642,0.0002


In [31]:
buscas_roubo, por_fold_roubo, linhas_roubo = {}, {}, []

In [32]:
for name, feats in FEATURES.items():
        print(f"buscando: {name}")
        b = buscar(train_val, "risco_roubo", feats, param_dist, cv)
        t = metricas_cv(b, train_val, "risco_roubo", feats, cv)
        buscas_roubo[name], por_fold_roubo[name] = b, t
        linhas_roubo.append({"modelo": name,
                       "R2": t["R2"].mean(), "MAE": t["MAE"].mean(),
                       "RMSE": t["RMSE"].mean(), "RMSE_desvio_folds": t["RMSE"].std()})

buscando: com_ano
buscando: sem_ano


In [33]:
base_roubo = baseline_cv(train_val, "risco_roubo", cv)
por_fold_roubo["media_celula"] = base_roubo
linhas_roubo.append({"modelo": "media_celula",
                   "R2": base_roubo["R2"].mean(), "MAE": base_roubo["MAE"].mean(),
                   "RMSE": base_roubo["RMSE"].mean(), "RMSE_desvio_folds": base_roubo["RMSE"].std()})

In [34]:
pd.DataFrame(linhas_roubo).round(4)

,modelo,R2,MAE,RMSE,RMSE_desvio_folds
0,com_ano,0.9360,0.0466,0.0778,0.0015
1,sem_ano,0.9326,0.0478,0.0799,0.0021
2,media_celula,0.9352,0.0470,0.0783,0.0021


In [35]:
CHOOSEN = {"risco_furto": "com_ano", "risco_roubo": "com_ano"}

In [36]:
models = {}
models["risco_furto"] = {
    "estimador": buscas_furto["com_ano"].best_estimator_,
    "features": FEATURES["com_ano"]
}
models["risco_roubo"] = {
    "estimador": buscas_roubo["com_ano"].best_estimator_,
    "features": FEATURES["com_ano"]
}

In [37]:
parts = []
for tr, va in cv:
    dv = train_val.iloc[va][["h3", "DATA_AUX", "risco"]].copy()
    for alvo, m in models.items():
        est = clone(m["estimador"]).fit(train_val.iloc[tr][m["features"]], train_val.iloc[tr][alvo])
        dv["pred_" + alvo] = est.predict(train_val.iloc[va][m["features"]])
    parts.append(dv)
oof = pd.concat(parts, ignore_index=True)
 
COLS = ["pred_risco_furto", "pred_risco_roubo"]

In [38]:
reg = LinearRegression(positive=True).fit(oof[COLS], oof["risco"])
w_furto, w_roubo = reg.coef_
print(f"peso furto: {w_furto:.3f} | peso roubo: {w_roubo:.3f} | intercepto: {reg.intercept_:.3f}")

peso furto: 0.700 | peso roubo: 0.353 | intercepto: 0.018


In [39]:
def combine(d, mode):
    f, r = d["pred_risco_furto"], d["pred_risco_roubo"]
    if mode == "media_simples":
        return (f + r) / 2
    if mode == "maximo":
        return np.maximum(f, r)
    if mode == "pesos_aprendidos":
        return np.clip(reg.predict(d[COLS]), 0, 1)
    raise ValueError(mode)

In [40]:
MODOS = ["media_simples", "maximo", "pesos_aprendidos"]

In [41]:
linhas = {}
for modo in MODOS:
    linhas[modo] = metricas(oof["risco"], combine(oof, modo))
    
pd.DataFrame(linhas).T.round(4)

,R2,MAE,RMSE
media_simples,0.9430,0.0490,0.0750
maximo,0.9534,0.0449,0.0678
pesos_aprendidos,0.9555,0.0492,0.0663


In [42]:
buscas_crime, por_fold_crime, linhas_crime = {}, {}, []

In [43]:
for name, feats in FEATURES.items():
        print(f"buscando: {name}")
        b = buscar(train_val, "risco", feats, param_dist, cv)
        t = metricas_cv(b, train_val, "risco", feats, cv)
        buscas_crime[name], por_fold_crime[name] = b, t
        linhas_crime.append({"modelo": name,
                       "R2": t["R2"].mean(), "MAE": t["MAE"].mean(),
                       "RMSE": t["RMSE"].mean(), "RMSE_desvio_folds": t["RMSE"].std()})

buscando: com_ano
buscando: sem_ano


In [44]:
base_crime = baseline_cv(train_val, "risco", cv)
por_fold_crime["media_celula"] = base_crime
linhas_crime.append({"modelo": "media_celula",
                   "R2": base_crime["R2"].mean(), "MAE": base_crime["MAE"].mean(),
                   "RMSE": base_crime["RMSE"].mean(), "RMSE_desvio_folds": base_crime["RMSE"].std()})

In [45]:
pd.DataFrame(linhas_crime).round(4)

,modelo,R2,MAE,RMSE,RMSE_desvio_folds
0,com_ano,0.9599,0.0410,0.0629,0.0005
1,sem_ano,0.9585,0.0415,0.0640,0.0005
2,media_celula,0.9601,0.0407,0.0627,0.0006


In [46]:
r = oof["pred_risco_furto"].corr(oof["pred_risco_roubo"])
print("correlação:", round(r, 3), "| VIF:", round(1 / (1 - r**2), 2))

correlação: 0.971 | VIF: 17.22
